In [ ]:
import pathlib
import obspy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import hvsrpy
from hvsrpy.sesame import reliability, clarity
from pathlib import Path
from scipy.interpolate import interp1d
from scipy.signal import find_peaks
from scipy.stats import skew
import joblib

plt.style.use(hvsrpy.HVSRPY_MPL_STYLE)

In [ ]:
elevation_in_m = 3.25
elevation_1500m_avg_in_m = -4.320

file_path = pathlib.Path(
    "D:/Document/Kuliah/Bismillah TA/Data/DATA 2 (Meulaboh)/"
    "Raw waveform microtremor data recorded in the City of Meulaboh/"
    "GL1-GEOBIT-60minutes.mseed"
)

if not file_path.exists():
    raise FileNotFoundError(f"File {file_path} not found.")

print("File exists.")

In [ ]:
stream = obspy.read(str(file_path))

if len(stream) != 3:
    raise ValueError(
        f"Recording must contain exactly 3 components (Z, N, E), "
        f"but found {len(stream)}."
    )

start_times = [tr.stats.starttime for tr in stream]
end_times   = [tr.stats.endtime for tr in stream]

common_start = max(start_times)
common_end   = min(end_times)
common_duration = common_end - common_start

if common_duration <= 0:
    raise ValueError("No overlapping time window among components.")

stream.trim(common_start, common_end)

print(f"Common duration: {common_duration / 60:.2f} minutes")

# Save trimmed file
output_path = file_path.with_name(file_path.stem + "_updated.mseed")
stream.write(str(output_path), format="MSEED")

print(f"Trimmed file saved as:\n{output_path}")

In [ ]:
preprocessing_settings = hvsrpy.settings.HvsrPreProcessingSettings()
preprocessing_settings.detrend = "constant"
preprocessing_settings.window_length_in_seconds = 20
preprocessing_settings.orient_to_degrees_from_north = 0.0
preprocessing_settings.filter_corner_frequencies_in_hz = (0.5, 10)
preprocessing_settings.ignore_dissimilar_time_step_warning = False

print("Preprocessing Summary")
print("-"*60)
preprocessing_settings.psummary()

In [ ]:
processing_settings = hvsrpy.settings.HvsrTraditionalProcessingSettings()
processing_settings.window_type_and_width = ("tukey", 0.1)
processing_settings.smoothing=dict(operator="konno_and_ohmachi",
                                   bandwidth=40,
                                   center_frequencies_in_hz=np.geomspace(0.5, 10, 128))
processing_settings.method_to_combine_horizontals = "squared_average"
processing_settings.handle_dissimilar_time_steps_by = "frequency_domain_resampling"

print("Processing Summary")
print("-"*60)
processing_settings.psummary()

In [ ]:
updated_file = output_path

srecords = hvsrpy.read([str(updated_file)])

srecords_preprocessed = hvsrpy.preprocess(srecords, preprocessing_settings)
hvsr = hvsrpy.process(srecords_preprocessed, processing_settings)

In [ ]:
# Cox et al. (2020) | Frequency-Domain Window Rejection Algorithm
n = 1.4
search_range_in_hz = (None, None)
_ = hvsrpy.frequency_domain_window_rejection(hvsr, n=n, search_range_in_hz=search_range_in_hz)

# STA-LTA | Short term average - Long term average rejection algorithm
#srecords = hvsrpy.read(file_path)
#srecords_preprocessed = hvsrpy.preprocess(srecords, preprocessing_settings)
#_ = hvsrpy.sta_lta_window_rejection(srecords_preprocessed, hvsr=hvsr)
#hvsrpy.maximum_value_window_rejection(srecords_preprocessed, hvsr=hvsr)

# Max Value | Maximum value window rejection
#srecords = hvsrpy.read(file_path)
#srecords_preprocessed = hvsrpy.preprocess(srecords, preprocessing_settings)
#_ = hvsrpy.maximum_value_window_rejection(srecords_preprocessed, hvsr=hvsr)

# Manual | Manual value window rejection
%matplotlib tk
_ = hvsrpy.manual_window_rejection(hvsr)

In [ ]:
%matplotlib inline
mfig, axs = hvsrpy.plot_pre_and_post_rejection(srecords_preprocessed, hvsr)
plt.show()

# =========================
# WINDOW STATISTICS
# =========================

# jumlah window = panjang boolean mask
total_windows = len(hvsr.valid_window_boolean_mask)

# jumlah window valid (True)
valid_windows = np.sum(hvsr.valid_window_boolean_mask)

# rejected
rejected_windows = total_windows - valid_windows

print("\nWindow Summary:")
print("-"*25)
print(f"Total window   : {total_windows}")
print(f"Valid window   : {valid_windows}")
print(f"Rejected window: {rejected_windows}")

In [ ]:
save_figure = False
save_results = False
file_path_prefix = "example_mhvsr_traditional_window_rejection"

if save_figure:
    file_path = f"{file_path_prefix}_all_panels.png"
    mfig.savefig(file_path)
    plt.close()
    print(f"Figure saved successfully to {file_path}!")

if save_results:
    file_path = f"{file_path_prefix}.csv"
    hvsrpy.object_io.write_hvsr_object_to_file(hvsr, file_path)
    print(f"Results saved successfully to {file_path}!")

In [ ]:
print("\nStatistical Summary:")
print("-"*20)
hvsrpy.summarize_hvsr_statistics(hvsr)
(sfig, ax) = hvsrpy.plot_single_panel_hvsr_curves(hvsr)
ax.get_legend().remove()
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
plt.show()

search_range_in_hz = (None, None)
verbose = 1
rejection_mask = hvsr.valid_window_boolean_mask.copy()
hvsr.update_peaks_bounded(search_range_in_hz=search_range_in_hz)
hvsr.valid_window_boolean_mask = rejection_mask & hvsr.valid_peak_boolean_mask
hvsr.valid_peak_boolean_mask   = rejection_mask & hvsr.valid_peak_boolean_mask

print("\nSESAME (2004) Clarity and Reliability Criteria:")
print("-"*47)
reliability(
    windowlength=preprocessing_settings.window_length_in_seconds,
    passing_window_count=int(np.sum(hvsr.valid_window_boolean_mask)),
    frequency=hvsr.frequency,
    mean_curve=hvsr.mean_curve(distribution="lognormal"),
    std_curve=hvsr.std_curve(distribution="lognormal"),
    search_range_in_hz=search_range_in_hz,
    verbose=verbose,
)
clarity(
    frequency=hvsr.frequency,
    mean_curve=hvsr.mean_curve(distribution="lognormal"),
    std_curve=hvsr.std_curve(distribution="lognormal"),
    fn_std=hvsr.std_fn_frequency(distribution="normal"),
    search_range_in_hz=search_range_in_hz,
    verbose=verbose,
)

In [ ]:
# =========================
# PREDIKSI Vs30 (MODEL LOKAL - 20 TITIK)
# =========================
fn_mean = hvsr.mean_fn_frequency(distribution="lognormal")
an_mean = hvsr.mean_fn_amplitude(distribution="lognormal")
TPI     = elevation_in_m - elevation_1500m_avg_in_m

# Susun DataFrame fitur -- URUTAN & NAMA KOLOM HARUS SAMA PERSIS
# dengan FEATURE_COLUMNS saat training model lokal:
# ["f0_hvsr", "a0_hvsr", "TPI", "elevation"]
X_new = pd.DataFrame([{
    "f0_hvsr"  : fn_mean,
    "a0_hvsr"  : an_mean,
    "TPI"      : TPI,
    "elevation": elevation_in_m
}])

print("\nFeature Matrix (model lokal):")
print(X_new)

# Load model lokal hasil training 20 titik
local_model_path = "xgboost_model_lokal_meulaboh.joblib" 
local_model = joblib.load(local_model_path)

# Prediksi Vs30 -- TIDAK di-exp() karena target training TIDAK di-log-transform
Vs30_pred_local = local_model.predict(X_new[["f0_hvsr", "a0_hvsr", "TPI", "elevation"]])

print("\nHasil Prediksi Vs30 (Model Lokal - XGBoost 20 Titik):")
print("-"*50)
print(f"Vs30 prediksi: {Vs30_pred_local[0]:.1f} m/s")

vs30_modelwise_df = pd.DataFrame({
    "Model"     : ["XGBoost Lokal (20 titik)"],
    "Vs30 (m/s)": [round(Vs30_pred_local[0], 1)]
})
print(vs30_modelwise_df)